In [162]:
import pandas as pd

In [163]:
#df = pd.read_csv("saudi_job_market.csv", engine='python', on_bad_lines='skip')
df = pd.read_csv("saudi_job_market.csv",engine="python",on_bad_lines="warn")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46919 entries, 0 to 46918
Data columns (total 49 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   scrape_id                             46919 non-null  object 
 1   jobid                                 46919 non-null  object 
 2   company_name                          46836 non-null  object 
 3   date_posted_parsed                    45207 non-null  object 
 4   job_title                             46919 non-null  object 
 5   description_text                      46498 non-null  object 
 6   benefits                              20258 non-null  object 
 7   qualifications                        0 non-null      float64
 8   job_type                              20955 non-null  object 
 9   location                              46919 non-null  object 
 10  salary_formatted                      20794 non-null  object 
 11  company_rating 

In [164]:
# Delete the columns related to scraping since most of them are null and do not add much information.
# Other columns have a lot of nulls and other problems and don't add much information or deserve the effort
cols_to_drop = [
    "scrape_id",
    "date_posted_parsed",
    "benefits",
    "qualifications",
    "job_type",
    "salary_formatted",
    "company_rating",
    "company_reviews_count",
    "country",
    "job_description_formatted",
    "logo_url",
    "shift_schedule",
    "timestamp",
    "requested_timestamp",
    "warning",
    "error",
    "error_code",
    "warning_code",
    "screenshot",
    "html",
    "page_id",
    "job_id",
    "collector_id",
    "collector_queue",
    "reparse_file",
    "input_url",
    "input_discovery_input_country",
    "input_discovery_input_domain",
    "input_discovery_input_keyword_search",
    "input_discovery_input_location",
    "scrape_date",
    "load_timestamp",
    "region",
    "company_link",
    "company_website",
    "domain",
    "apply_link",
    "srcname",
    "url",
    "is_expired",
    "discovery_input",
    "date_posted",
    # description_text and description are the same so we will drop one of them
    'description_text',
    # job_location and location are the same so we will drop one of them
    'job_location'
]
df.drop(columns=cols_to_drop, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46919 entries, 0 to 46918
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   jobid         46919 non-null  object
 1   company_name  46836 non-null  object
 2   job_title     46919 non-null  object
 3   location      46919 non-null  object
 4   description   46498 non-null  object
dtypes: object(5)
memory usage: 1.8+ MB


In [165]:
# Renaming columns to match th schema
df.rename(columns={
    'jobid': 'job_id',
    'company_name': 'company',
    'description' : 'job_description'
}, inplace=True)

In [166]:
description_nulls = (df['job_description'].isna().sum() / len(df)) * 100
print(f"{description_nulls:.2f}%")
# Doesn't seem to effect too much so we will drop it
df = df.dropna(subset=['job_description'])

0.90%


In [167]:
company_nulls = (df['company'].isna().sum() / len(df)) * 100
print(f"{company_nulls:.2f}%")
# Doesn't seem to effect too much so we will drop it
df = df.dropna(subset=['company'])

0.18%


In [168]:
# Dropping duplicates in jobid and description
df = df.drop_duplicates(subset=['job_id'])
df = df.drop_duplicates(subset=['job_description'])

In [169]:
df['job_title'] = df['job_title'].str.strip()
df['company'] = df['company'].str.strip()

df['job_title'] = df['job_title'].str.lower()
df['company'] = df['company'].str.lower()

In [170]:
# Standardizing locations names
location_mapping = {

    # =========================
    # Countries
    # =========================
    "المملكة العربية السعودية": "Saudi Arabia",

    # =========================
    # Regions
    # =========================
    "منطقة القصيم": "Qassim Region",
    "Al Qaṣīm": "Qassim Region",

    "منطقة الشرقية": "Eastern Province",
    "Ash Sharqīyah": "Eastern Province",

    "منطقة عسير": "Asir Region",
    "`Asīr": "Asir Region",

    "منطقة الرياض": "Riyadh Region",
    "منطقة الباحة": "Al Bahah Region",
    "منطقة جازان": "Jazan Region",

    # =========================
    # Major Cities
    # =========================
    "الرياض": "Riyadh",
    "riyadh": "Riyadh",

    "جدة": "Jeddah",
    "jeddah": "Jeddah",

    "الدمام": "Dammam",
    "dammam": "Dammam",

    "مكة": "Mecca",
    "makkah": "Mecca",
    "mecca": "Mecca",

    "المدينة": "Medina",
    "Al Madīnah": "Medina",

    # =========================
    # Standardizing Spellings
    # =========================
    "جازان": "Jizan",
    "Jīzan": "Jizan",

    "الباحة": "Al Bahah",
    "Al Bāhah": "Al Bahah",

    "الهفوف": "Al Hufuf",
    "Al Hufūf": "Al Hufuf",

    "رابغ": "Rabigh",
    "Rābigh": "Rabigh",

    "الخفجي": "Al Khafji",
    "Al Khafjī": "Al Khafji",

    # =========================
    # Other Cities
    # =========================
    "تبوك": "Tabuk",
    "الظهران": "Dhahran",
    "الخبر": "Al Khobar",
    "الجبيل": "Jubail",
    "الخرج": "Al-Kharj",
    "الطائف": "Ta'if",
    "طريف": "Turaif",
    "حسي": "Al Hassi",
    "أبها": "Abha",

    "رأس الخير": "Ras Al-Khair",
    "بقيق": "Buqayq",
    "خميس مشيط": "Khamis Mushait",
    "الدرعية": "Ad-Diriyah",
    "المجمعة": "Al Majma",

    "ينبع البحر": "Yanbu' al Bahr",
    "ثول": "Thuwal",
    "حفر الباطن": "Hafar Al-Batin",
    "نجران": "Najran",
    "الوجه": "Al Wajh",
    "رأس تنورة": "Ras Tanura",

    "العلا": "Al Ula",
    "بريدة": "Buraidah",
    "القطيف": "Qatif",
    "ضبا": "Duba",

    "صبيا": "Sabya",
    "وادي الدواسر": "Wadi Al Dawasir",
    "عنيزة": "Unaizah",
    "القريات": "Al Qurayyat",
    "السليل": "As Sulayyil",
}

In [171]:
df['location_cleaned'] = (df['location'].str.strip().replace(location_mapping))

In [172]:
# Create an empty skills column to be populated later
df['skills'] = None

In [173]:
# Reordering for easier reading
df = df[
    ['job_id',
     'job_title',
     'job_description',
     'company',
     'location',
     'location_cleaned',
     'skills']
]

In [174]:
# Final table
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14261 entries, 0 to 46868
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   job_id            14261 non-null  object
 1   job_title         14261 non-null  object
 2   job_description   14261 non-null  object
 3   company           14261 non-null  object
 4   location          14261 non-null  object
 5   location_cleaned  14261 non-null  object
 6   skills            0 non-null      object
dtypes: object(7)
memory usage: 891.3+ KB


In [175]:
df.head()

,job_id,job_title,job_description,company,location,location_cleaned,skills
0,32ecfd63823c0691,lead storage backup engineer,Job Description: Essential Job Functions: ...,dxc technology,الرياض,Riyadh,None
1,cbda9008da66fbe2,software engineer,"TAM is a Saudi publicly listed company, specia...",tam development co.,الرياض,Riyadh,None
2,7623b2d1848b53b6,senior storage backup engineer,Job Description: Responsibilities: St...,dxc technology,الرياض,Riyadh,None
3,0728eeb152762255,critical environment field service engineer,Business Function Overview: In alignment ...,microsoft,الدمام,Dammam,None
4,f2ad4ea17dd2c1c1,l3 network security engineer,"As an L3 Network Security Engineer at SWATX, y...",swatx,Riyadh,Riyadh,None


In [176]:
# @title
#from google.colab import files

#df.to_csv('saudi_job_market_cleaned.csv', index=False, encoding='utf-8-sig')

#files.download('saudi_job_market_cleaned.csv')

In [177]:
# TODO: Merge job_title and job_description into a single text field for LLM-based skill extraction .